# 개별종목 조합D — LogisticRegression

`기본모델/01.LogisticRegression.ipynb`과 같은 `models.logistic.build_logistic_baseline`을 가져오고
조합D 피처를 주입합니다. 기본모델 코드는 `models/`에 한 번만 존재합니다.
후보·라벨·날짜 그룹 12폴드 실행은 모든 조합이 같은 공통 함수를 사용합니다.


In [1]:
# 1. 기본모델을 가져옵니다.
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display

project_root = Path.cwd().resolve()
while project_root != project_root.parent and not (project_root / "pyproject.toml").is_file():
    project_root = project_root.parent
if not (project_root / "pyproject.toml").is_file():
    raise RuntimeError("프로젝트 루트를 찾지 못했습니다.")
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from models.logistic import build_logistic_baseline  # noqa: E402

MODEL_NAME = 'LogisticRegression'
MODEL_BUILDER = build_logistic_baseline


In [2]:
# 2. 조합D의 피처 값만 지정합니다.
import json

COMBINATION = 'D'
FEATURE_COLUMNS = (
    'atr_ratio',
    'hv_20',
    'range_1',
    'range_20',
    'bb_bandwidth',
    'volume_z_20',
    'turnover_20',
    'log_amihud_20',
)

report_path = project_root / "reports" / "stock_feature_combinations.json"
report = json.loads(report_path.read_text(encoding="utf-8"))
combination_report = report["combinations"][COMBINATION]
panel = combination_report["panel"]
print("학습 기간:", panel["first_date"], "~", panel["last_date"])
print("학습 행·종목:", panel["model_rows"], panel["stocks"])
print(f"조합{COMBINATION} 피처:", FEATURE_COLUMNS)

folds = pd.DataFrame(combination_report["outer_fold_results"])
model_folds = folds.loc[folds["model"].eq(MODEL_NAME)].reset_index(drop=True)
fold_columns = [
    "fold",
    "selected_class_weight",
    "train_dates",
    "valid_start",
    "valid_end",
    "accuracy",
    "training_majority_baseline_accuracy",
    "accuracy_minus_training_majority_baseline",
    "macro_f1",
    "down_recall",
    "core_harmonic_mean",
]
display(model_folds.loc[:, fold_columns].round(4))

metric_columns = [
    "accuracy",
    "training_majority_baseline_accuracy",
    "accuracy_minus_training_majority_baseline",
    "macro_f1",
    "down_recall",
    "core_harmonic_mean",
]
display(model_folds.loc[:, metric_columns].mean().to_frame("OOS 폴드 평균").round(4))

# 24개 노트북이 각각 중복 학습하지 않도록 실제 fit은 공통 실행기에서 한 번 수행합니다.
print("재실행 명령: python scripts/run_stock_model_experiment.py")


학습 기간: 20100331 ~ 20240822
학습 행·종목: 171557 162
조합D 피처: ('atr_ratio', 'hv_20', 'range_1', 'range_20', 'bb_bandwidth', 'volume_z_20', 'turnover_20', 'log_amihud_20')


,fold,selected_class_weight,train_dates,valid_start,valid_end,accuracy,training_majority_baseline_accuracy,accuracy_minus_training_majority_baseline,macro_f1,down_recall,core_harmonic_mean
0,1,balanced,750,20130410,20130705,0.3878,0.3701,0.0177,0.3071,0.0726,0.1529
1,2,balanced,999,20140414,20140711,0.4762,0.4741,0.0021,0.2972,0.0872,0.1772
2,3,balanced,1248,20150421,20150716,0.3685,0.3330,0.0356,0.3636,0.2325,0.3072
3,4,balanced,1496,20160422,20160719,0.4070,0.4128,-0.0058,0.3282,0.1359,0.2332
4,5,balanced,1745,20170424,20170721,0.4168,0.4182,-0.0014,0.3150,0.1796,0.2693
5,6,balanced,1994,20180503,20180731,0.3988,0.3912,0.0075,0.3926,0.3130,0.3637
6,7,balanced,2243,20190514,20190806,0.4364,0.4615,-0.0252,0.3409,0.0989,0.1956
7,8,balanced,2492,20200518,20200807,0.3719,0.3144,0.0575,0.3392,0.1989,0.2813
8,9,balanced,2741,20210518,20210810,0.4033,0.4423,-0.0389,0.3604,0.2058,0.2966
9,10,balanced,2989,20220519,20220812,0.3557,0.3343,0.0213,0.3363,0.1392,0.2313


,OOS 폴드 평균
accuracy,0.4002
training_majority_baseline_accuracy,0.3851
accuracy_minus_training_majority_baseline,0.0151
macro_f1,0.3435
down_recall,0.1749
core_harmonic_mean,0.2594


재실행 명령: python scripts/run_stock_model_experiment.py
